from cli.build_ragpack import run_pipeline_v12

source_files = sorted(p for p in INPUT_DIR.iterdir()
                      if p.is_file() and p.suffix.lower() in {".txt", ".md"})
if GGUF_PATH is None:
    print("No GGUF — re-run the GGUF cell first.")
elif not source_files:
    print(f"No .txt/.md sources in {INPUT_DIR}.")
    print("Upload files and convert PDFs first, then re-run this cell.")
else:
    result = run_pipeline_v12(
        input_dir=INPUT_DIR,
        output_dir=OUTPUT_DIR,
        gguf_path=GGUF_PATH,
        chunk_size=CHUNK_SIZE,
        overlap=OVERLAP,
        creation_time=CREATION_TIME,
        verbose=True,
    )

    print(f"\n\u2705 Build complete.")
    print(f"   files:  {result.file_count}")
    print(f"   chunks: {result.chunk_count}")
    print(f"   output: {result.output_dir}")

In [ ]:
import sys

IS_COLAB = "google.colab" in sys.modules
print(f"Environment: {'Colab' if IS_COLAB else 'Local Jupyter'}")
print(f"Python: {sys.version.split()[0]}")

## Install dependencies

In **Colab** both lines below run (they are Colab-only). The pipeline already
pins `llama-cpp-python>=0.3.0` in its `pyproject.toml`, so the second install
is an idempotent no-op kept explicit for clarity. In **local** mode nothing is
installed — `pip install -e .` from the repo root is assumed done already.

In [ ]:
# Colab-only: install the pipeline from GitHub (main branch) + llama-cpp-python.
# Locally: assume the pipeline is already installed in the active env.
if IS_COLAB:
    !pip install -q git+https://github.com/rag-fish/noesisnoema-pipeline.git@main  # Colab-only
    !pip install -q llama-cpp-python  # Colab-only; idempotent (already a pipeline dep)
else:
    print("Local mode — using the installed pipeline package "
          "(pip install -e . from the repo root).")

## Obtain the embedder GGUF

Three ways to point at the GGUF, handled by the single cell below:

- **Option A (Colab, recommended for repeat runs):** mount Google Drive and
  read the GGUF from a Drive path.
- **Option B (Colab, one-shot):** if the Drive path is missing, upload the
  GGUF directly via the file chooser.
- **Option C (local):** set `GGUF_PATH` to a local filesystem path.

The cell **validates** the file is a real GGUF (magic bytes). If you
accidentally upload a *document* here (`.pdf`/`.txt`/`.md`), it is moved
into the input folder automatically — no re-upload needed — and you just
re-run this cell with the actual GGUF.

In [ ]:
import os
from pathlib import Path

INPUT_DIR = Path("/content/input") if IS_COLAB else Path("./input")  # shared with the source cells

def looks_like_gguf(path: Path) -> bool:
    # Real GGUF files start with the 4-byte magic b"GGUF".
    try:
        with open(path, "rb") as f:
            return f.read(4) == b"GGUF"
    except OSError:
        return False

GGUF_PATH = None
if IS_COLAB:
    # OPTION A: mount Drive (recommended for repeat runs)
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        candidate = Path("/content/drive/MyDrive/Models/nomic-embed-text-v1.5.Q5_K_M.gguf")
        if candidate.is_file():
            GGUF_PATH = candidate
    except Exception:
        pass
    # OPTION B: direct upload (one-shot)
    if GGUF_PATH is None:
        from google.colab import files
        print("Drive path not found — upload the embedder GGUF "
              "(nomic-embed-text-v1.5.Q5_K_M.gguf):")
        try:
            uploaded = files.upload()
        except Exception:
            uploaded = {}
        if uploaded:
            GGUF_PATH = Path("/content") / next(iter(uploaded.keys()))
        else:
            print("Nothing uploaded.")
else:
    # OPTION C: local path — adjust as needed
    GGUF_PATH = Path("/path/to/nomic-embed-text-v1.5.Q5_K_M.gguf")

# Validate: must exist AND carry the GGUF magic bytes.
if GGUF_PATH is not None and GGUF_PATH.is_file() and looks_like_gguf(GGUF_PATH):
    print(f"GGUF OK: {GGUF_PATH}  ({GGUF_PATH.stat().st_size / 1024**2:.1f} MB)")
else:
    if GGUF_PATH is not None and GGUF_PATH.is_file():
        print(f"NOT A GGUF: {GGUF_PATH.name} — this cell needs the embedder *model*, not a document.")
        if GGUF_PATH.suffix.lower() in {".pdf", ".txt", ".md"}:
            INPUT_DIR.mkdir(exist_ok=True)
            dest = INPUT_DIR / GGUF_PATH.name
            GGUF_PATH.rename(dest)
            print(f"It looks like a source document — moved it to {dest},")
            print("where the Source documents cell will pick it up. No re-upload needed.")
    GGUF_PATH = None
    print("\nRe-run this cell and provide the GGUF (or fix the Drive path) before continuing.")

## Verify the GGUF SHA-256 fingerprint

Critical identity check (ADR-0011 §3). The app validates `embedder.model_hash`
on import, and that hash is the SHA-256 of the GGUF file. This cell **does not
halt on mismatch** — you may be deliberately building for a future/experimental
embedder — it just prints the result clearly.

In [ ]:
import hashlib

EXPECTED_FINGERPRINT = "0c7930f6c4f6f29b7da5046e3a2c0832aa3f602db3de5760a95f0582dbd3d6e6"

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

if GGUF_PATH is None:
    print("No GGUF yet — re-run the GGUF cell first.")
else:
    actual = sha256_file(GGUF_PATH)
    print(f"SHA-256:  {actual}")
    print(f"Expected: {EXPECTED_FINGERPRINT}")
    if actual == EXPECTED_FINGERPRINT:
        print("\u2705 Fingerprint matches the NoesisNoema app embedder.")
    else:
        print("\u26a0\ufe0f  Fingerprint does NOT match. The resulting pack will be rejected "
              "by NoesisNoema v0.4+ unless the app is rebuilt with this GGUF.")

## Source documents (.txt / .md / .pdf)

Upload your source files with the **single chooser** in the next cell — it
accepts `.txt`, `.md`, **and** `.pdf` in one go. The pipeline ingests
`.txt`/`.md` directly; if you upload `.pdf`, run the optional conversion cell
below to turn them into `.txt` first.

Pressing **Cancel** in the chooser is perfectly fine — it just keeps whatever
files are already in the input directory.

In [ ]:
INPUT_DIR = Path("/content/input") if IS_COLAB else Path("./input")
INPUT_DIR.mkdir(exist_ok=True)

if IS_COLAB:
    # Single chooser for all formats. Cancel is normal, not an error.
    from google.colab import files
    print("Upload .txt / .md / .pdf files "
          "(or press Cancel to keep existing files).")
    try:
        uploaded = files.upload()
    except Exception:
        uploaded = {}
    if uploaded:
        for name, data in uploaded.items():
            (INPUT_DIR / name).write_bytes(data)
    else:
        print("No new files uploaded — keeping existing contents.")
else:
    print(f"Local mode — place .txt/.md/.pdf files in {INPUT_DIR.resolve()} "
          "and re-run this cell to list them.")

all_files = sorted(p for p in INPUT_DIR.iterdir() if p.is_file())
print(f"\nFiles in {INPUT_DIR} ({len(all_files)}):")
for p in all_files:
    print(f"  {p.name}  ({p.stat().st_size / 1024:.1f} KB)")

txt_md = [p for p in all_files if p.suffix.lower() in {".txt", ".md"}]
pdfs = [p for p in all_files if p.suffix.lower() == ".pdf"]
if txt_md:
    print(f"\nREADY: {len(txt_md)} source file(s) (.txt/.md).")
elif pdfs:
    print("\nPDFs detected — run the next cell to convert.")
else:
    print("\nNo usable files yet — upload and re-run this cell.")

## (Optional) PDF → txt pre-conversion

If you have `.pdf` files, run this cell to convert them to `.txt` next to the
originals; with no PDFs present it **skips quietly**. Uses `pypdf`
(lightweight, pure-Python). Converted `.txt` files are picked up
automatically and re-listed below — no need to re-run the upload cell.

In [ ]:
# OPTIONAL: convert any .pdf files in INPUT_DIR to .txt next to them.
# Uses pypdf (lightweight, pure-Python). Adjust as needed.
def convert_pdfs_to_txt(directory: Path) -> int:
    pdfs = list(directory.glob("*.pdf"))
    if not pdfs:
        print("No PDFs to convert — skipping.")
        return 0
    try:
        from pypdf import PdfReader
    except ImportError:
        !pip install -q pypdf
        from pypdf import PdfReader
    for pdf_path in pdfs:
        reader = PdfReader(str(pdf_path))
        text = "\n\n".join((page.extract_text() or "") for page in reader.pages)
        out_path = pdf_path.with_suffix(".txt")
        out_path.write_text(text, encoding="utf-8")
        print(f"  {pdf_path.name} → {out_path.name}  ({len(text):,} chars)")
    return len(pdfs)

n = convert_pdfs_to_txt(INPUT_DIR)
print(f"Converted {n} PDF(s).")

# Re-list the ready sources here so you don't need to re-run the upload cell.
txt_md = sorted(p for p in INPUT_DIR.iterdir()
                if p.is_file() and p.suffix.lower() in {".txt", ".md"})
if txt_md:
    print(f"READY: {len(txt_md)} source file(s) (.txt/.md).")
    for p in txt_md:
        print(f"  {p.name}  ({p.stat().st_size / 1024:.1f} KB)")
else:
    print("WARNING: still no .txt/.md sources — "
          "upload files and re-run the upload cell.")

## Configuration

**Note on reproducibility:** `creation_time` feeds the deterministic `pack_id`
derivation (`_derive_pack_id` in `cli/build_ragpack.py`). Using `datetime.now()`
below means the `pack_id` changes on every run. To get a **reproducible** pack
(same sources + embedder + config + time → same `pack_id`), pin `CREATION_TIME`
to a fixed string instead, e.g. `CREATION_TIME = "2026-06-10T00:00:00Z"`.

In [ ]:
from datetime import datetime, timezone

OUTPUT_DIR = Path("/content/output") if IS_COLAB else Path("./output")
OUTPUT_DIR.mkdir(exist_ok=True)

CHUNK_SIZE    = 512    # tokens per chunk
OVERLAP       = 50     # token overlap between chunks
CREATION_TIME = datetime.now(timezone.utc).replace(microsecond=0).isoformat().replace("+00:00", "Z")

print(f"chunk_size:    {CHUNK_SIZE}")
print(f"overlap:       {OVERLAP}")
print(f"creation_time: {CREATION_TIME}")
print(f"output_dir:    {OUTPUT_DIR}")

## Build the pack

Calls the canonical `run_pipeline_v12()` — the exact function the CLI invokes
for `--embedder llama-cpp`. The first call loads the GGUF (a few seconds on CPU).
If this cell prints a warning instead of building, fix the inputs and re-run —
the inspect/zip cells below all depend on a successful build.

In [ ]:
from cli.build_ragpack import run_pipeline_v12

source_files = sorted(p for p in INPUT_DIR.iterdir()
                      if p.is_file() and p.suffix.lower() in {".txt", ".md"})
if not source_files:
    print(f"No .txt/.md sources in {INPUT_DIR}.")
    print("Upload files and convert PDFs first, then re-run this cell.")
else:
    result = run_pipeline_v12(
        input_dir=INPUT_DIR,
        output_dir=OUTPUT_DIR,
        gguf_path=GGUF_PATH,
        chunk_size=CHUNK_SIZE,
        overlap=OVERLAP,
        creation_time=CREATION_TIME,
        verbose=True,
    )

    print(f"\n✅ Build complete.")
    print(f"   files:  {result.file_count}")
    print(f"   chunks: {result.chunk_count}")
    print(f"   output: {result.output_dir}")

## Inspect the manifest

Visually verify the manifest is well-formed for the app:

- `pack_version == "1.2"`
- `embedder.model_hash` matches the expected fingerprint (Cell 5)
- `embedder.embedding_dimension == 768`
- `embedder.pooling == "mean"`
- `embedder.l2_normalized == true`

In [ ]:
import json

if "result" not in globals():
    print("No build result yet — run the Build cell successfully first.")
else:
    with open(result.written_paths["manifest"]) as f:
        manifest = json.load(f)

    print(json.dumps(manifest, indent=2, ensure_ascii=False))

## Inspect embeddings shape

Verify shape is `(N, 768)`, dtype `float32`, and the first-row norm ≈ 1.0
(rows are L2-normalized).

In [ ]:
import numpy as np

if "result" not in globals():
    print("No build result yet — run the Build cell successfully first.")
else:
    arr = np.load(result.written_paths["embeddings"])
    print(f"embeddings.npy shape: {arr.shape}")
    print(f"dtype:                {arr.dtype}")
    print(f"first row norm:       {np.linalg.norm(arr[0]):.6f}")
    print(f"first 8 values:       {arr[0, :8]}")

## Inspect chunks + citations

`chunks.json` holds the chunk text and metadata; `citations.jsonl` holds one
citation record per line. `chunk_index` aligns with the `embeddings.npy` row.

In [ ]:
if "result" not in globals():
    print("No build result yet — run the Build cell successfully first.")
else:
    with open(result.written_paths["chunks"]) as f:
        chunks = json.load(f)

    print(f"Total chunks: {len(chunks)}\n")
    print("First chunk:")
    print(json.dumps(chunks[0], indent=2, ensure_ascii=False)[:500])
    print("…")

    # Citations
    print("\nFirst 3 citations:")
    with open(result.written_paths["citations"]) as f:
        for i, line in enumerate(f):
            if i >= 3:
                break
            print(json.dumps(json.loads(line), indent=2, ensure_ascii=False)[:300])
            print("---")

## Zip + offer download

Archives the output directory into a `<source-name>_ragpack.zip` (named after the
first source document, sanitized). In Colab the browser download is triggered
automatically; locally the path is printed.

In [ ]:
import re
import shutil

if "result" not in globals():
    print("No build result yet — run the Build cell successfully first.")
else:
    source_files = sorted(p for p in INPUT_DIR.iterdir()
                          if p.is_file() and p.suffix.lower() in {".txt", ".md"})

    stem = source_files[0].stem if source_files else OUTPUT_DIR.name
    safe = re.sub(r"[^A-Za-z0-9._-]", "_", stem)[:40].rstrip("._-") or OUTPUT_DIR.name
    zip_base = OUTPUT_DIR.parent / f"{safe}_ragpack"

    zip_path = Path(shutil.make_archive(str(zip_base), "zip", str(OUTPUT_DIR)))
    print(f"✅ Zipped: {zip_path}  ({zip_path.stat().st_size / 1024**2:.2f} MB)")

    if IS_COLAB:
        from google.colab import files
        files.download(str(zip_path))  # triggers browser download
        print("Downloading…")
    else:
        print(f"Local mode — zip available at {zip_path.resolve()}")

## (Optional) Save to Drive

In Colab, copies the zip into `MyDrive/Ragpacks` for safekeeping. Skipped locally.

In [ ]:
if "zip_path" not in globals():
    print("No zip yet — run the Zip cell first.")
elif IS_COLAB:
    drive_out = Path("/content/drive/MyDrive/Ragpacks")
    drive_out.mkdir(parents=True, exist_ok=True)
    shutil.copy(zip_path, drive_out / zip_path.name)
    print(f"✅ Saved to Drive: {drive_out / zip_path.name}")
else:
    print("Local mode — Drive save skipped.")

## Next steps & troubleshooting

**Import into NoesisNoema:** Settings ▸ Manage RAGpacks ▸ Import .zip, then
select the downloaded `output.zip`.

**If the app rejects the pack**, the alert names the exact validation that
failed — adjust the GGUF or rebuild the app accordingly:

- `embedderFingerprintMismatch` — the GGUF SHA-256 ≠ the app's expected hash
  (see Cell 5). Use the matching GGUF, or rebuild the app with this GGUF.
- `embeddingDimensionMismatch` — embeddings aren't `(N, 768)`; wrong embedder.
- Other validations map 1:1 to the manifest fields checked in Cell 9.

**Verify retrieval quality:** run the 5-question RAG verification UAT recorded
in ADR-0011 / past transcripts against the imported pack.